# BRD4 BD1 and BD2 Y-randomization
Actual CHPC scores, 26 November 2026. Original baseline is index 0; indexes 1–200 are shuffled labels. Run from the repository root or this folder. Install `figentra`, `numpy`, `pandas`, and `matplotlib` before running. The notebook validates transferred Q² values against the CHPC summaries and exports 600 dpi PNG, PDF and SVG.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from figentra import apply_style, style_axes, save_figure

base = Path.cwd()
if not (base / "BD1_Q2.csv").exists():
    base = base / "CHPC_Y_RANDOM" / "results"
summary = pd.read_csv(base / "Y_randomization_summary.csv").set_index("domain")
frames = {d: pd.read_csv(base / f"{d}_Q2.csv") for d in ("BD1", "BD2")}
for d, f in frames.items():
    assert f.permutation.tolist() == list(range(201))
    q = f.Q2.iloc[1:]
    s = summary.loc[d]
    b = int((q >= f.Q2.iloc[0]).sum())
    actual = [f.Q2.iloc[0], q.mean(), q.std(ddof=1), q.min(), q.max(), q.quantile(.95), (b+1)/201]
    expected = s[["original_Q2", "random_Q2_mean", "random_Q2_sd", "random_Q2_min", "random_Q2_max", "random_Q2_95th_percentile", "p_value_one_sided"]].to_numpy(float)
    np.testing.assert_allclose(actual, expected, atol=1e-12, rtol=1e-12)
display(summary)


In [ ]:
apply_style()
colors = {"BD1": "#2563A6", "BD2": "#D66A28"}
bins = np.arange(-.17, .031, .01)
ymax = int(np.ceil((max(np.histogram(f.Q2.iloc[1:], bins)[0].max() for f in frames.values())+12)/10)*10)
fig, axs = plt.subplots(1, 2, figsize=(260/25.4, 100/25.4))
for i, (ax, (d, f)) in enumerate(zip(axs, frames.items())):
    r, o = f.iloc[1:], f.iloc[0]
    ax.hist(r.Q2, bins=bins, color=colors[d], edgecolor="white", linewidth=.7)
    ax.axvline(r.Q2.mean(), color="#202020", lw=1.7, ls="--")
    ax.axvline(0, color="#202020", lw=1.3, ls=":")
    ax.set(xlim=(-.17, .03), ylim=(0, ymax), xlabel="Randomized Q²", ylabel="Frequency")
    ax.set_xticks([-.15, -.10, -.05, 0])
    ax.yaxis.set_major_locator(MultipleLocator(20))
    ax.text(0, 1.055, f"({chr(65+i)}) {d}", transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax.text(.97, .96, f"Original Q² = {o.Q2:.4f}\np = {summary.loc[d, 'p_value_one_sided']:.6f}", transform=ax.transAxes, ha="right", va="top", fontsize=11, fontweight="bold", bbox=dict(facecolor="white", edgecolor="none", pad=2.5))
    style_axes(ax)
fig.subplots_adjust(left=.087, right=.981, bottom=.20, top=.86, wspace=.36)
save_figure(fig, "Figure_01_Y_randomization_histogram", output_dir=base/"regenerated_figures")
plt.show()


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(260/25.4, 100/25.4))
for i, (ax, (d, f)) in enumerate(zip(axs, frames.items())):
    r, o = f.iloc[1:], f.iloc[0]
    ax.scatter(r.permutation, r.Q2, s=12, c=colors[d], alpha=.83, linewidths=0, zorder=3)
    ax.axhline(o.Q2, color="#202020", linewidth=2, linestyle="--", zorder=2)
    ax.axhline(0, color="#777777", linewidth=1, linestyle=":", zorder=1)
    ax.set(xlim=(0, 205), ylim=(-.20, .85), xlabel="Permutation index", ylabel="Q²")
    ax.set_xticks([0,50,100,150,200])
    ax.set_yticks([-.2,0,.2,.4,.6,.8])
    ax.text(0, 1.055, f"({chr(65+i)}) {d}", transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom")
    ax.text(.98, o.Q2+.035, f"Original Q² = {o.Q2:.4f}", transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=11, fontweight="bold")
    ax.text(.98, .50, f"p = {summary.loc[d, 'p_value_one_sided']:.6f}", transform=ax.get_yaxis_transform(), ha="right", fontsize=11, fontweight="bold")
    style_axes(ax)
fig.subplots_adjust(left=.087, right=.981, bottom=.20, top=.86, wspace=.36)
save_figure(fig, "Figure_02_Y_randomization_scatter", output_dir=base/"regenerated_figures")
plt.show()
